# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [ ]:
import os
import sys

# 1. 항상 최신 코드를 받기 위해 기존 폴더를 지우고 새로 Clone (본인 fork)
%cd /content
!rm -rf 2026-HYU-AUE8088-PA2
!git clone https://github.com/Jeong-dahun/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/2026-HYU-AUE8088-PA2

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ImageNet 사전학습 가중치 로드 (분석 2용)
# 출처: facebookresearch/deit — DeiT-Small/16, ImageNet-1k 사전학습.
#   DeiT-S 는 ViT-S/16 과 구조가 동일(dim=384, depth=12, heads=6)하여 백본 가중치를 그대로 매핑할 수 있다.
#   (timm/torchvision import 가 아니라 .pth 파일만 내려받으므로 금지사항 위반이 아님)
import torch

DEIT_URL = "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth"

def load_pretrained_into(model):
    """DeiT-S 체크포인트를 우리 ViT 구현체에 로드한다. 매칭/누락 키 수를 출력."""
    sd = torch.hub.load_state_dict_from_url(DEIT_URL, map_location="cpu")
    sd = sd.get("model", sd)
    remapped = {}
    for k, v in sd.items():
        if k.startswith("head"):          # 1000-class 분류기 → 버림 (MultiTaskHead 는 random init 유지)
            continue
        # 우리 MLP 는 nn.Sequential 이라 fc1/fc2 → 0/3 으로 키 이름만 변환
        k = k.replace("mlp.fc1", "mlp.0").replace("mlp.fc2", "mlp.3")
        remapped[k] = v
    missing, unexpected = model.load_state_dict(remapped, strict=False)
    print(f"[pretrained] matched={len(remapped)} keys  "
          f"missing={list(missing)}  unexpected={list(unexpected)}")
    return model

# 단독 점검: missing 은 head.* (task head) 만, unexpected 는 비어 있어야 정상.
_ = load_pretrained_into(vit_small_patch16_224().to(device))

In [ ]:
def train_vit(use_pretrained: bool, epochs: int = 25):
    """ViT-S/16 을 scratch 또는 ImageNet-pretrained 초기화로 학습한다."""
    set_seed(SEED, deterministic=True)
    model = vit_small_patch16_224().to(device)
    if use_pretrained:
        load_pretrained_into(model)

    optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    tag = "vit_s16-" + ("pretrained" if use_pretrained else "scratch")

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level2-{tag}",
        config={
            "backbone": "vit_s16", "pretrained": use_pretrained,
            "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
        },
        tags=WANDB_TAGS + [tag],
    )
    trainer = MultiTaskTrainer(model, optim, sched, losses, device, TrainConfig(epochs=epochs), logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history, "pretrained": use_pretrained},
               f"../checkpoints/level2_{tag}.pth")
    return model, history


# 1) Scratch 학습 (분석 1·3 의 ViT 기준선)
vit_scratch, vit_scratch_hist = train_vit(use_pretrained=False, epochs=25)
print("Scratch  Final Avg-MF1:", vit_scratch_hist["val_avg_mf1"][-1])
print("Scratch  per-attribute MF1:", vit_scratch_hist["val_per_mf1"][-1])

## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.

In [ ]:
# === [분석 2 준비] ImageNet pretrained ViT 학습 =============================
# scratch 학습이 끝난 뒤 실행. 같은 epoch 예산으로 사전학습 초기화 효과를 측정한다.
vit_pre, vit_pre_hist = train_vit(use_pretrained=True, epochs=25)
print("Pretrained  Final Avg-MF1:", vit_pre_hist["val_avg_mf1"][-1])
print("Pretrained  per-attribute MF1:", vit_pre_hist["val_per_mf1"][-1])

In [ ]:
# === [분석 1] CNN vs Transformer: ResNet-50 vs ViT-S(scratch) ==============
import pandas as pd

def load_hist(path):
    return torch.load(path, map_location="cpu")["history"]

def final_row(h):
    per = h["val_per_mf1"][-1]
    return {"weather": per["weather"], "scene": per["scene"],
            "timeofday": per["timeofday"], "Avg-MF1": h["val_avg_mf1"][-1]}

rows = {}
if os.path.exists("../checkpoints/level1_resnet50.pth"):
    rows["ResNet-50 (Lv1)"] = final_row(load_hist("../checkpoints/level1_resnet50.pth"))
else:
    print("⚠️ level1_resnet50.pth 없음 — Level 1 에서 ResNet-50 학습 후 다시 실행하세요.")
rows["ViT-S scratch (Lv2)"] = final_row(load_hist("../checkpoints/level2_vit_s16-scratch.pth"))

df1 = pd.DataFrame(rows).T[["weather", "scene", "timeofday", "Avg-MF1"]].round(4)
print(df1)
df1.to_csv("../checkpoints/level2_cnn_vs_vit.csv")

In [ ]:
# === [분석 2] Pretrained vs Scratch (ViT-S) ================================
import pandas as pd, matplotlib.pyplot as plt

def _row(h):
    per = h["val_per_mf1"][-1]
    return {"weather": per["weather"], "scene": per["scene"],
            "timeofday": per["timeofday"], "Avg-MF1": h["val_avg_mf1"][-1]}

sc = torch.load("../checkpoints/level2_vit_s16-scratch.pth",    map_location="cpu")["history"]
pr = torch.load("../checkpoints/level2_vit_s16-pretrained.pth", map_location="cpu")["history"]

df2 = pd.DataFrame({"scratch": _row(sc), "pretrained": _row(pr)}).T
df2 = df2[["weather", "scene", "timeofday", "Avg-MF1"]].round(4)
print(df2)
print("Avg-MF1 향상 (pretrained - scratch):",
      round(pr["val_avg_mf1"][-1] - sc["val_avg_mf1"][-1], 4))

plt.figure(figsize=(7, 5))
plt.plot(range(1, len(sc["val_avg_mf1"]) + 1), sc["val_avg_mf1"], marker="o", ms=3, label="scratch")
plt.plot(range(1, len(pr["val_avg_mf1"]) + 1), pr["val_avg_mf1"], marker="o", ms=3, label="pretrained")
plt.xlabel("epoch"); plt.ylabel("val Avg-MF1")
plt.title("ViT-S: scratch vs ImageNet(DeiT) pretrained")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("../checkpoints/level2_pretrained_vs_scratch.png", dpi=120); plt.show()

In [ ]:
# === [분석 3] 속성별 거동: ViT vs ResNet-50 (per-attr MF1 + confusion matrix) ===
import matplotlib.pyplot as plt
from src.models.resnet import resnet50
from src.utils.metrics import per_attribute_macro_f1

# 비교용 ViT: pretrained 가 있으면 그것, 없으면 scratch
vit_model = vit_pre if "vit_pre" in globals() else vit_scratch
vit_pred, _, vit_tgt, _ = collect_predictions(vit_model, val_loader, device)

# ResNet-50 (Level 1 체크포인트 로드)
r50 = resnet50().to(device)
r50.load_state_dict(torch.load("../checkpoints/level1_resnet50.pth", map_location=device)["state_dict"])
r50_pred, _, r50_tgt, _ = collect_predictions(r50, val_loader, device)

vit_mf1 = per_attribute_macro_f1(vit_pred, vit_tgt)
r50_mf1 = per_attribute_macro_f1(r50_pred, r50_tgt)
print("속성별 Macro-F1")
print(f"{'attr':<12}{'ResNet-50':>12}{'ViT-S':>10}")
for a in ATTRIBUTES:
    print(f"{a:<12}{r50_mf1[a]:>12.4f}{vit_mf1[a]:>10.4f}")

# weather / timeofday confusion matrix 비교 (오류 분포 차이 관찰 → 리포트에 가설 서술)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for col, (name, pred, tgt) in enumerate(
        [("ResNet-50", r50_pred, r50_tgt), ("ViT-S", vit_pred, vit_tgt)]):
    cms = confusion_matrices(pred, tgt)
    for row, a in enumerate(["weather", "timeofday"]):
        ax = axes[row][col]
        ax.imshow(cms[a], vmin=0, vmax=1, cmap="Blues")
        ax.set_title(f"{name} — {a}")
        ax.set_xticks(range(len(CLASS_NAMES[a]))); ax.set_xticklabels(CLASS_NAMES[a], rotation=45, ha="right")
        ax.set_yticks(range(len(CLASS_NAMES[a]))); ax.set_yticklabels(CLASS_NAMES[a])
        for i in range(cms[a].shape[0]):
            for j in range(cms[a].shape[1]):
                ax.text(j, i, f"{cms[a][i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.savefig("../checkpoints/level2_cm_compare.png", dpi=120); plt.show()